In [ ]:
import cv2
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets


def load_labels(label_path):
    with open(label_path, "r") as f:
        content = f.read().strip()
    return [int(ch) for ch in content if ch.isdigit()]


def get_total_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return total


def read_frame(video_path, frame_idx):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()

    if not ret:
        raise RuntimeError(f"Cannot read frame {frame_idx} from {video_path}")

    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def clamp(value, low, high):
    return max(low, min(value, high))


def viewer_advanced(video_path, label_path):
    labels = load_labels(label_path)
    total_frames = get_total_frames(video_path)
    total_labels = len(labels)

    print(f"Frames: {total_frames}")
    print(f"Labels: {total_labels}")

    slider = widgets.IntSlider(
        value=0,
        min=0,
        max=total_frames - 1,
        step=1,
        description="Frame:",
        continuous_update=False,
        layout=widgets.Layout(width="700px")
    )

    frame_box = widgets.BoundedIntText(
        value=0,
        min=0,
        max=total_frames - 1,
        step=1,
        description="Go to:",
        layout=widgets.Layout(width="220px")
    )

    btn_prev_100 = widgets.Button(description="-100")
    btn_prev_10 = widgets.Button(description="-10")
    btn_prev_1 = widgets.Button(description="-1")
    btn_next_1 = widgets.Button(description="+1")
    btn_next_10 = widgets.Button(description="+10")
    btn_next_100 = widgets.Button(description="+100")
    btn_first = widgets.Button(description="First")
    btn_last = widgets.Button(description="Last")

    output = widgets.Output()
    updating = {"busy": False}

    def draw_frame(frame_idx):
        with output:
            clear_output(wait=True)

            frame = read_frame(video_path, frame_idx)
            label = labels[frame_idx] if frame_idx < total_labels else "N/A"

            if label == "N/A":
                color = "orange"
            else:
                color = "green" if label == 0 else "red"

            plt.figure(figsize=(7, 5))
            plt.imshow(frame)
            plt.title(
                f"Frame: {frame_idx} / {total_frames - 1} | Label: {label}",
                color=color,
                fontsize=14
            )
            plt.axis("off")
            plt.show()

            diff = total_labels - total_frames
            if diff == 0:
                status = "PERFECT MATCH"
            elif diff > 0:
                status = f"EXTRA LABELS: {diff}"
            else:
                status = f"EXTRA FRAMES: {-diff}"

            print(f"Video frames: {total_frames}")
            print(f"Labels      : {total_labels}")
            print(f"Status      : {status}")

    def set_frame(new_value):
        new_value = clamp(new_value, 0, total_frames - 1)

        if updating["busy"]:
            return

        updating["busy"] = True
        slider.value = new_value
        frame_box.value = new_value
        draw_frame(new_value)
        updating["busy"] = False

    def on_slider_change(change):
        if change["name"] == "value" and not updating["busy"]:
            updating["busy"] = True
            frame_box.value = change["new"]
            draw_frame(change["new"])
            updating["busy"] = False

    def on_box_change(change):
        if change["name"] == "value" and not updating["busy"]:
            updating["busy"] = True
            slider.value = change["new"]
            draw_frame(change["new"])
            updating["busy"] = False

    def move(delta):
        set_frame(slider.value + delta)

    btn_prev_100.on_click(lambda _: move(-100))
    btn_prev_10.on_click(lambda _: move(-10))
    btn_prev_1.on_click(lambda _: move(-1))
    btn_next_1.on_click(lambda _: move(1))
    btn_next_10.on_click(lambda _: move(10))
    btn_next_100.on_click(lambda _: move(100))
    btn_first.on_click(lambda _: set_frame(0))
    btn_last.on_click(lambda _: set_frame(total_frames - 1))

    slider.observe(on_slider_change, names="value")
    frame_box.observe(on_box_change, names="value")

    controls_row1 = widgets.HBox([
        btn_first, btn_prev_100, btn_prev_10, btn_prev_1,
        btn_next_1, btn_next_10, btn_next_100, btn_last
    ])

    controls_row2 = widgets.HBox([slider, frame_box])

    display(controls_row1)
    display(controls_row2)
    display(output)

    draw_frame(0)


video_path = "/root/.cache/kagglehub/datasets/manith04/gdgdfhfg/versions/1/004/004_noglasses_mix.mp4"
label_path = "/root/.cache/kagglehub/datasets/manith04/gdgdfhfg/versions/1/004/004_noglasses_mixing_head.txt"


viewer_advanced(video_path, label_path)